### Rebuilds `openalex.works.work_authorships` in full each run (Walden End to End workflow)

Full rebuild every run (oxjob #660): reads the current work_authors / affiliation-MV /
profile state for every work, so enrichment staleness self-corrects nightly by
construction. Replaces the former watermark + change-detection intake, the
MV-institution bolt-on, and the oxjob #582 clear-empty step (subsumed by the
empty-row branch). ~6 min serverless XL measured for the full build.

One seat per person (oxjob #1861): a seat that repeats an earlier one (the whole list repeated, same name and
author id, or the same author id in another script) is dropped after author ids are bound by position, and its
affiliations merge into the seat it repeats.

Catalogue strings (oxjob #1912): a byline name that is a library-catalogue string ('Washington, George,\n1732-1799.', LoC URIs, MARC 880 tags) is served as the name it holds; the fixes cell builds the lookup each run.

Identifier seats (oxjob #1925): a seat whose name is only an ORCID or Lattes link (or a bare ORCID iD) is not a person; step 5b folds it into the named seat it follows, handing on its ORCID where the byline's shape was checked against ORCID's records.

### Step 1: Create enriched authors

In [ ]:
REFRESH MATERIALIZED VIEW openalex.works.work_author_affiliations_mv

In [ ]:
%run ../utils/variables

In [ ]:
-- oxjob #1912: the name a library-catalogue byline string holds. 652K works showed raw catalogue strings as
-- author names ('Washington, George,\n    1732-1799.', id.loc.gov URIs, MARC 880 tags, 'Porat, Dan 1964-').
-- Source of truth and tests: oxjobs #1912 work/byline_names.py (this body is generated from it by build_udf_sql.py).
-- oxjob #1944 (30 Oct 2026): date words start a word ('René 1715-1791' kept 'Re'), wide spaces inside a name are a
-- space ('Silva, Matheus de   Oliveira'), BnF partial years and runs of roles, ';' inside parentheses, descriptors
-- after the comma ('Billot, médecin de l'Empereur'), the old BnF form 'Sourdis (Charles d'Escoubleau, marquis de)'.
-- Source of truth and tests now: oxjobs #1944 work/cleaner/byline_names.py (193 cases). MatchAuthors' mint calls
-- this function too (#1944): it names a profile minted from a catalogue string with the name it holds.
-- oxjob #1952 (2 Nov 2026): the mint's first-comma flip of a BnF heading, as 2023 profiles and source records hold it
-- ('Jean . Auteur du texte Audouze' -> 'Jean Audouze'), BnF's paired 'Auteur du texte / Autrice du texte', more BnF
-- roles (Destinataire de lettres, Réalisateur, Scénariste, Metteur en scène, Auteur ou responsable intellectuel),
-- 'd'' joined to its surname ('Gustave d'Espinay'), a term of address kept when there is no forename ('abbé Fangouse').
-- Source of truth and tests now: oxjobs #1952 work/cleaner/byline_names.py (221 cases).
-- Revert: remove this cell, the next one, and the three #1912 edits in the main statement (#1952 alone: put back
-- the body of walden bef8796c; #1944 alone: the body of walden 07b3bbdc).
CREATE OR REPLACE FUNCTION openalex.authors.clean_byline_name(s STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #1912: the name a library-catalogue byline string holds (MARC name fields: subfield a kept, d dates, e roles, 0/1 URIs, t titles, 880 tags dropped, control characters and mojibake fixed as #1910 does for titles). Returns its input when there is nothing to fix or no name to keep. Source + tests: oxjobs #1912, #1944, #1952 work/cleaner/byline_names.py'
AS $$
import html
import re
import unicodedata

_MAX_ROUNDS = 4  # corpus has strings mojibaked up to ~5 levels deep, each round strictly shrinks

_CP1252_REV = {}
for _b in range(0x80, 0xA0):
    try:
        _CP1252_REV[bytes([_b]).decode("cp1252")] = _b
    except UnicodeDecodeError:
        pass  # 0x81 0x8D 0x8F 0x90 0x9D are undefined in cp1252

_CORE_LEADS_2 = set("ÃÂÄÅÌÎÏÐÑ")  # Ì (CC) covers NFD combining-mark mojibake ("MuÌnster")
_PERIPHERAL_LEADS_2 = set("ÆÇÒÓÔÕÖØÙÚÛÜ")


def _is_cont(ch):
    cp = ord(ch)
    if 0x80 <= cp <= 0xBF:
        return True
    return _CP1252_REV.get(ch, 0) >= 0x80  # cp1252 specials are all 0x80-0x9F


def _detect(s):
    if not s:
        return False
    n = len(s)
    for i, ch in enumerate(s):
        if ch in _CORE_LEADS_2 and i + 1 < n and _is_cont(s[i + 1]):
            return True
        if ch == "â" and i + 2 < n and _is_cont(s[i + 1]) and _is_cont(s[i + 2]):
            return True
        cp = ord(ch)
        if (0xE0 <= cp <= 0xEF and cp != 0xE2 and i + 2 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2])
                and not (s[i + 1] == "\xa0" and s[i + 2] == "\xa0")):
            return True
        if (0xF0 <= cp <= 0xF4 and i + 3 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2]) and _is_cont(s[i + 3])
                and not (s[i + 1] == s[i + 2] == s[i + 3] == "\xa0")):
            return True
    return False


def _fallback_signature(s, codec):
    if not s:
        return False

    def byte_of(ch):
        try:
            return ch.encode(codec)[0]
        except (UnicodeEncodeError, IndexError):
            return None

    n = len(s)
    for i, ch in enumerate(s):
        b = byte_of(ch)
        if b is not None and 0xC2 <= b <= 0xDF and b != ord(ch) and i + 1 < n:
            b2 = byte_of(s[i + 1])
            if b2 is not None and 0x80 <= b2 <= 0xBF:
                return True
    return False


def _to_bytes_unified(s):
    out = bytearray()
    for ch in s:
        cp = ord(ch)
        if cp <= 0xFF:
            out.append(cp)
        else:
            b = _CP1252_REV.get(ch)
            if b is None:
                return None
            out.append(b)
    return bytes(out)


def _validate(s, raw):
    if raw is None:
        return None
    try:
        fixed = raw.decode("utf-8", errors="strict")
    except UnicodeDecodeError:
        return None
    if len(fixed) >= len(s) or "�" in fixed:
        return None
    return fixed


def _badness(s):
    score = 0
    for ch in s:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            score += 3
        elif cat.startswith("S"):
            score += 1
    return score


def _repair_once(s):
    candidates = []
    if _detect(s):
        fixed = _validate(s, _to_bytes_unified(s))
        if fixed is not None:
            candidates.append(fixed)
    for codec in ("iso-8859-2", "cp1250"):
        if _fallback_signature(s, codec):
            try:
                raw = s.encode(codec, errors="strict")
            except UnicodeEncodeError:
                continue
            fixed = _validate(s, raw)
            if fixed is not None:
                candidates.append(fixed)
    if not candidates:
        return None
    return min(candidates, key=_badness)  # ties keep candidate order (min is stable)


def _fix_bare_c1(s):
    if not any(0x80 <= ord(ch) <= 0x9F for ch in s):
        return s
    out = []
    for ch in s:
        if 0x80 <= ord(ch) <= 0x9F:
            try:
                ch = bytes([ord(ch)]).decode("cp1252")
            except UnicodeDecodeError:
                pass  # 0x81 0x8D 0x8F 0x90 0x9D undefined -> keep
        out.append(ch)
    return "".join(out)


def repair_mojibake(s):
    if not s:
        return s
    if s.isascii():
        return s
    cur = s
    for _ in range(_MAX_ROUNDS):
        nxt = _repair_once(cur)
        if nxt is None:
            break
        cur = nxt
    if cur is not s:
        cur = _fix_bare_c1(cur)
    return cur


CTRL = re.compile(r"[\x00-\x1f\x7f-\x9f]")
_MARC_PAIR = re.compile(r"\x98([^\x98\x9c]{0,40}?)\x9c")
_SEPARATORS = re.compile(r"[\t\n\v\f\r\x1c-\x1f]+")
_OTHER_CTRL = re.compile(r"[\x00-\x08\x0e-\x1b\x7f-\x9f]")
_CP1252_SPECIALS = set("€‚ƒ„…†‡ˆ‰Š‹ŒŽ"
                       "‘’“”•–—˜™š›œžŸ")
_PUNCT = {0x85, 0x91, 0x92, 0x93, 0x94, 0x95, 0x96, 0x97, 0x99}   # ... ' ' " " bullet en-dash em-dash TM
_LOW_QUOTES = {0x82, 0x84}                                      # single/double low-9 quotation marks
_ANGLE_QUOTES = {0x8B, 0x9B}                                    # single angle quotation marks
_DAGGERS = {0x86, 0x87}
_UPPER = {0x8A, 0x8C, 0x8E, 0x9F}                               # S-caron, OE, Z-caron, Y-diaeresis
_LOWER = {0x9A, 0x9C, 0x9E}                                     # s-caron, oe, z-caron
_QUOTES = {0x82, 0x84, 0x8B, 0x91, 0x92, 0x93, 0x94, 0x9B}
_ASCII_QUOTES = ("\"", "'")
_IN_WORD = {0x92, 0x96, 0x97, 0x9A, 0x9C, 0x9E}               # apostrophe, en/em dash, s-caron, oe, z-caron
_LIGATURE_TAIL = re.compile(r"(?:cations?|cial|cien|nanc|nal|nit|nd|rst|eld|gur|xed|tness|ber|lm|uen|uid|ux|ow|ect|er\b|ers\b|ed\b)")
_CLOSING = re.compile("[\x91-\x94\u2018\u2019\u201c\u201d\"']")


def _ok_before(prev):
    if prev == "":
        return True
    cp = ord(prev)
    if 0x20 <= cp <= 0x7E:
        return True
    return cp > 0xFF and prev not in _CP1252_SPECIALS and not (0x80 <= cp <= 0x9F)


def _is_opening(prev):
    return prev == "" or prev.isspace() or prev in "([{/-"


def _cp1252_char(b, prev, nxt, nxt2, rest, closes_later):
    if not _ok_before(prev) or "\x80" <= nxt <= "\x9f":
        return ""  # only a lone C1 char is read: a run of them ('Parentesco\x99\x83\x8e\x8b') is garbage
    ch = bytes([b]).decode("cp1252", errors="ignore")
    p_letter, n_letter = prev.isalpha(), nxt.isalpha()
    if p_letter and n_letter:
        if b not in _IN_WORD or _LIGATURE_TAIL.match(rest):
            return ""
    if b == 0x95:  # bullet: a list separator, never between two letters (PDF 'ff' slot: 'E\x95ects')
        return ch if not (p_letter and n_letter) else ""
    if b in _PUNCT:
        return ch
    if b in _LOW_QUOTES:  # an opening quote, and only when a closing quote follows ('\x82e Human' is a 'Th' slot)
        return ch if _is_opening(prev) and nxt.isalnum() and closes_later else ""
    if b in _ANGLE_QUOTES:
        return ch if not (p_letter and n_letter) else ""
    if b in _DAGGERS:
        return ch if not n_letter else ""
    if b == 0x80:  # euro sign: only beside a number, or after 'M'/'k' that follows a number or space
        if nxt.isdigit() or (nxt == " " and nxt2.isdigit()) or prev.isdigit():
            return ch
        return ""
    if b == 0x89:  # per mille: only after a digit
        return ch if prev.isdigit() else ""
    if b in _UPPER:  # word-initial capital before a lowercase letter, or inside an all-caps word
        if not p_letter and (nxt.islower() or (nxt.isupper() and nxt2.isupper())):
            return ch
        return ""
    if b == 0x9C:  # oe ligature: French words (oeuvre, coeur, soeur, noeud, Ricoeur) put it word-initially or after
        if (not p_letter or prev in "cnsvfmhCNSVFMH") and nxt.isascii() and nxt.islower():
            return ch
        return ""
    if b in _LOWER:  # s-caron, z-caron: inside or at the end of a word, or word-initial before a letter
        if n_letter or (p_letter and not n_letter):
            return ch
        return ""
    return ""  # 0x83 f-hook, 0x88 circumflex, 0x98 small tilde, and the five undefined bytes


def _map_c1(t):
    out = []
    n = len(t)
    for i, c in enumerate(t):
        b = ord(c)
        if 0x80 <= b <= 0x9F:
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < n else ""
            nxt2 = t[i + 2] if i + 2 < n else ""
            if b in _QUOTES and (nxt in _ASCII_QUOTES or prev in _ASCII_QUOTES):
                continue  # a C1 quote right beside an ASCII quote duplicates it
            out.append(_cp1252_char(b, prev, nxt, nxt2, t[i + 1:i + 9], bool(_CLOSING.search(t, i + 1))))
        else:
            out.append(c)
    return "".join(out)


def clean_title_controls(s):
    if not s or not isinstance(s, str) or not CTRL.search(s):
        return s
    t = repair_mojibake(s)              # 1 exact mojibake repair
    t = _MARC_PAIR.sub(r"\1", t)       # 2 MARC non-sort marks
    t = _map_c1(t)                      # 3 lone C1 -> its cp1252 character where it fits, else dropped
    t = _SEPARATORS.sub(" ", t)         # 4 separators -> space, every other control dropped
    t = _OTHER_CTRL.sub("", t)
    t = " ".join(t.split())
    return t if t else s


CTRL = re.compile(r"[\x00-\x1f\x7f-\x9f]")
URI = re.compile(r"(?i)(?:https?://|www\.|\bid\.loc\.gov|\bviaf\.org/|\borcid\.org/|\blattes\.cnpq\.br/)")
MARC880 = re.compile(r"(?:^|\s)880-\d{2}(?:/\S*)?\.?(?=\s|$)")
DATES_END = re.compile(r"\d{3,4}\??\s*-\s*(?:\d{3,4}\??)?\s*[.,)]?\s*$")
WIDE = re.compile(r"\S\s{3,}\S")
IDORCID = re.compile(r"(?i)\bid_orcid\b|(?<![\d-])\d{4}-\s?\d{4}-\d{4}-\d{3}[\dX](?![\d-])")
FLIPPED_DATES = re.compile(r"^[^\d,\n]+?\s\d{3,4}\??\s*-\s*(?:\d{3,4}\??)?\s+[^\W\d][^\s\d,]*$")
_BNF_ROLE_WORDS = ("Auteur", "Éditeur", "Traducteur", "Illustrateur", "Préfacier", "Compositeur", "Librettiste", "Graveur",
                   "Photographe", "Cartographe", "Imprimeur", "Libraire", "Dessinateur", "Annotateur", "Commentateur",
                   "Rédacteur")
_BNF_AUTHOR_KIND = (r"(?:du texte|de lettres|de l['’]argument|du commentaire|de la musique|adapté|de l['’]idée originale|"
                    r"de la pièce jointe|de l['’]envoi|ou responsable intellectuel)")
_BNF_ROLE_ALT = (rf"(?:(?:Auteur|Autrice) {_BNF_AUTHOR_KIND}(?:\s*/\s*(?:Auteur|Autrice) {_BNF_AUTHOR_KIND})?|"
                 r"Destinataire de lettres|Destinataire de l['’]envoi|Éditeur scientifique|Imprimeur-libraire|Metteur en scène|"
                 + "|".join(w + r"(?!\s+(?:du|de|des|d['’]|en)\b)" for w in _BNF_ROLE_WORDS + ("Réalisateur", "Scénariste"))
                 + ")")
BNF_ROLE = re.compile(rf"(?i)\.\s*{_BNF_ROLE_ALT}(?:\.\s*{_BNF_ROLE_ALT})*\s*$")
FLIPPED_BNF = re.compile(rf"^(?P<fore>[^\W\d_][^,]*?)\s*\.\s*(?P<role>{_BNF_ROLE_ALT}(?:\s*\.\s*{_BNF_ROLE_ALT})*)"
                         rf"\s+(?P<sur>(?![/.])[^,]+?)\.?\s*$")
_ROLE_INSIDE = re.compile(rf"\.\s*{_BNF_ROLE_ALT}(?=[\s.]|$)")


_DEPARTEMENTS = frozenset((
    "Ain Aisne Allier Alpes-de-Haute-Provence Hautes-Alpes Alpes-Maritimes Ardèche Ardennes Ariège Aube Aveyron "
    "Bouches-du-Rhône Calvados Cantal Charente Charente-Maritime Cher Corrèze Corse Côte-d'Or Côtes-d'Armor Côtes-du-Nord "
    "Creuse Dordogne Doubs Drôme Eure Eure-et-Loir Finistère Gard Haute-Garonne Gers Gironde Hérault Ille-et-Vilaine Indre "
    "Indre-et-Loire Isère Jura Landes Loir-et-Cher Loire Haute-Loire Loire-Atlantique Loire-Inférieure Loiret Lot "
    "Lot-et-Garonne Lozère Maine-et-Loire Manche Marne Haute-Marne Mayenne Meurthe-et-Moselle Meuse Morbihan Moselle Nièvre "
    "Nord Oise Orne Pas-de-Calais Puy-de-Dôme Pyrénées-Atlantiques Basses-Pyrénées Hautes-Pyrénées Pyrénées-Orientales "
    "Bas-Rhin Haut-Rhin Rhône Haute-Saône Saône-et-Loire Sarthe Savoie Haute-Savoie Seine-Maritime Seine-Inférieure "
    "Seine-et-Marne Yvelines Deux-Sèvres Somme Tarn Tarn-et-Garonne Var Vaucluse Vendée Vienne Haute-Vienne Vosges Yonne "
    "Territoire-de-Belfort Essonne Hauts-de-Seine Seine-Saint-Denis Val-de-Marne Val-d'Oise Seine-et-Oise Guadeloupe "
    "Martinique Guyane Réunion").split())


def _flipped_bnf(t):
    m = FLIPPED_BNF.match(t) if "\n" not in t else None
    return m if (m and not m.group("sur")[:1].islower() and _LETTER.search(m.group("sur"))
                 and not _ROLE_INSIDE.search(m.group("sur"))) else None


def has_artefact(s):
    return bool(s) and bool(CTRL.search(s) or URI.search(s) or MARC880.search(s) or DATES_END.search(s)
                            or WIDE.search(s) or IDORCID.search(s) or BNF_ROLE.search(s) or FLIPPED_DATES.search(s)
                            or _flipped_bnf(s))


_YEAR = r"\d{3,4}\??"
_DATE_WORDS = r"(?:(?<![^\W\d_])(?:(?-i:b\.|d\.|ca\.|c\.|fl\.|n\.|m\.)|approximately|active|born|died|né|née|floruit)|-)"
DATE_SEG = re.compile(rf"(?i)^\(?\s*(?:{_DATE_WORDS}\s*)*(?:\d{{1,2}}\s+\w+\s+)?(?:{_YEAR}(?:\s*or\s*\d{{1,4}})?\s*-?\s*(?:{_YEAR}(?:\s*or\s*\d{{1,4}})?)?|"
                      rf"\d{{1,2}}(?:st|nd|rd|th)\s+(?:century|cent\.?)(?:\s*(?:B\.?\s?C|A\.?\s?D)\.?)?|"
                      rf"{_YEAR}\s*(?:or|-)\s*{_YEAR})\s*\)?\s*[.,\x3b:]*\s*$")
_YOR = rf"{_YEAR}(?:\s*or\s*\d{{1,4}})?"
INLINE_DATES = re.compile(rf"(?i)[,\s]*\(?\s*(?:{_DATE_WORDS}\s*)*(?:\d{{1,2}}\s+\w+\s+)?{_YOR}\s*-?\s*(?:{_YOR})?\s*\)?\s*[.,]?\s*$")
INLINE_CENTURY = re.compile(r"(?i)[,\s]+(?:active|fl\.)?\s*\d{1,2}(?:st|nd|rd|th)\s+(?:century|cent\.?)"
                            r"(?:\s*-\s*\d{1,2}(?:st|nd|rd|th)\s+(?:century|cent\.?))?\.?(?:\s*(?:B\.?\s?C|A\.?\s?D)\.?)?\s*$")
PAREN_DATES = re.compile(r"\s*\([^():]*(?:\d{3,4}|\d\d\.\.)[^():]*\)\.?\s*$")     # '(1674-1755)' '(American Artist, Active 1896-1943)'
FIRM_PAREN = re.compile(r"(?i)\((?:viuda|vda|herederos|hijos|sucesores|widow|heirs|sons|veuve|erben|witwe)[^)]*\)")
_RELATORS = (r"author|joint author|co-?author|editor|joint editor|ed|eds|edt|comp|compiler|joint compiler|translator|"
             r"joint translator|tr|trans|transl|trl|illustrator|ill|illus|ilu|composer|cmp|librettist|lyricist|arranger|"
             r"arr|former owner|fmo|contributor|ctb|sponsor|publisher|pbl|printer|prt|engraver|egr|photographer|pht|"
             r"cartographer|ctg|creator|cre|performer|prf|writer of preface|writer of introduction|author of introduction|"
             r"author of introduction, etc|writer of supplementary textual content|writer of added text|"
             r"commentator|annotator|ann|dedicatee|dte|lithographer|binder|bookseller|designer|artist|art|painter|"
             r"aut|redactor|reviser|director|narrator|singer|speaker|interviewer|interviewee|host|"
             r"degree supervisor|thesis advisor|ths|praeses|respondent|issuing body|isb|donor|dnr|addressee|"
             r"autor|editora?|coord|coordinator|pref|prologuista|traductor|ilustrador|"
             r"herausgeber|hrsg|verfasser|mitwirkender|übersetzer|"
             r"imp|impr|imprimeur|researcher|research assistant|"
             r"plaintiff|defendant|appellant|appellee|petitioner|respondent|complainant|libellant|intervenor|claimant|"
             r"plaintiff-appellee|plaintiff-appellant|defendant-appellee|defendant-appellant|plaintiff in error|"
             r"defendant in error|"
             r"ant\. pos|antiguo poseedor|dedicante|grabador|impresor|"
             r"auteur|auteur du texte|éditeur scientifique|traducteur|illustrateur|préfacier|"
             r"\(?μτφρ\.?\)?|\(?επιμ\.?\)?|"
             r"นักวิจัย|หัวหน้าโครงการวิจัย|หัวหน้าโครงการ|ผู้ช่วยนักวิจัย|ที่ปรึกษาโครงการ|ที่ปรึกษา|ผู้วิจัย|ผู้แต่ง|บรรณาธิการ")
RELATOR_SEG = re.compile(rf"(?i)^\(?\s*(?:(?:{_RELATORS})\.?\s*[,\x3b&]?\s*(?:and\s+)?)+\)?\s*[.,\x3b:]*\s*$")
INLINE_RELATOR = re.compile(rf"(?i),\s*(?:(?:{_RELATORS})\.?\s*[,\x3b&]?\s*(?:and\s+)?)+\s*$")
SUFFIX = re.compile(r"(?i)^(?:jr|sr|jun|sen|junior|senior|[ivx]{2,5}|\d(?:st|nd|rd|th))\.?,?$")   # 'I.' is an initial
_CORP_WORDS = re.compile(
    r"(?i)\b(?:univ\w*|universi\w+|college|school|institut\w*|commission\w*|department|dept|office|board|society|"
    r"soci[eé]t[eé]|library|biblioteca|bibliothek|council|association|associa\w+|committee|bureau|ministry|ministerio|"
    r"company|co|inc|ltd|corp\w*|survey|agency|academy|acad\w+|museum|foundation|church|congress|government|"
    r"administration|division|laboratory|laboratories|service|services|center|centre|centro|club|conference|"
    r"congresso|congreso|kongress|meeting|symposium|workshop|seminar|press|publishing|publishers|editorial|verlag|"
    r"group|team|consortium|network|project|programme|program|organization|organisation|union|federation|league|"
    r"parliament|senate|assembly|navy|army|military|states|nations|republic|kingdom|county|"
    r"province|national|international|royal|hospital|railroad|railway|firm|brothers|bros|sons|"
    r"empresa|fundação|fundación|prefeitura|secretaria|embrapa|"
    r"cuerpo|colegio|convento|monasterio|archivo|compañía|imprenta|cofradía|cabildo|iglesia|catedral|"
    r"bibliotecari\w*|archiver\w*|collection|sammlung|treaties|"
    r"gesellschaft|verein|ges|akademie|kommission|amt|ministère|comité|conseil|instituto|sociedad|asociación|"
    r"secretaría|dirección|junta|ayuntamiento|diputación|gobierno|universidade|faculdade|faculty|faculté|"
    r"facultad|gakkai|kyōkai|kenkyūkai|kenkyūjo|kyoku|jimusho|kaisha|"
    r"gmbh|srl|governor|courts?|legislature|magistrat|stadtmagistrat|ministerstvo|police|mayor|prefecture|"
    r"henshūjo|henshūbu|shuppanbu|jimukyoku|daigaku|gakkō|kenkyūkai|rigakubu|"
    r"dept|directorate|branch|section|unit|division|subcommittee|chapter|"
    r"paroisse|musée|mairie|abbaye|église|école|lycée|hôpital|syndicat|bibliothèque|archives)\b")   # French (#1944)
_ADDRESS = re.compile(r"(?i)^(?:madame|mme|mrs|mr|miss|mlle|ms|dr|sister|brother|father|mother|abbé|abbe|rev|sir|dame|lady|"
                      r"saint|st|frau|herr|fräulein|señora|señor|doña|don|pseud)\.?$")
_RANK = re.compile(r"(?i)^(?:the\s.*|queen|king|emperor|empress|prince|princess|duke|duchess|count|countess|conte|contessa|"
                   r"comte|comtesse|graf|gräfin|freiherr|freifrau|baron|baroness|baronne|marquis|marquise|marchese|earl|"
                   r"viscount|pope|bishop|archbishop|cardinal|sultan|shah|khan|caliph|tsar|czar|grand duke|elector|"
                   r"byzantine emperor|consort.*|.*\bking of\b.*|.*\bqueen of\b.*)\.?$")
_CJK = re.compile(r"[぀-ヿ㐀-䶿一-鿿가-힯豈-﫿]")
_LETTER = re.compile(r"[^\W\d_]")

_URL_TOKEN = r"(?:https?://|www\.)[^\s()]+|\b(?:id\.loc\.gov|viaf\.org|orcid\.org|lattes\.cnpq\.br)/[^\s()]*"
URL_AT_END = re.compile(rf"(?i)\s*[\x3b,|]?\s*(?:ORCID(?:\s*ID)?\s*:?\s*)?\[?(?:{_URL_TOKEN})\]?\s*$")
URL_PREFIX = re.compile(rf"(?i)(^|\x3b\s*)(?:ORCID(?:\s*ID)?\s*:?\s*)?\[?(?:{_URL_TOKEN})\]?\s*")
URL_MID = re.compile(rf"(?i)\s*[\x3b|]?\s*(?:ORCID(?:\s*ID)?\s*:?\s*)?\[?(?:{_URL_TOKEN})\]?\s*")
ID_ORCID = re.compile(r"(?i) ?[\x3b,|]? ?\(? ?(?:id_orcid|orcid)? ?:? ?#?(?<![\d-])\d{4}- ?\d{4}-\d{4}-\d{3}[\dX](?![\d-]) ?\)?")
EMAIL = re.compile(r"\s*[\x3b,]?\s*[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
REPEC = re.compile(r"(?i)\s*RePEc:\S+")
LABEL_ONLY = re.compile(r"(?i)^\W*(?:orcid(?:\s*id)?|contacts?|e-?mail|url|web\s*site|website|homepage|lattes|gnd)\W*$")
SHELFMARK = re.compile(r"[A-Z]{2,4}(?: [A-Z]{1,4}){1,3} [\w().]*\d[\w().]*\.?")
HOLDING_CODE = re.compile(r"[A-Z]{1,6}(?:-[A-Za-z]{1,6})+|\d{5,}")          # MARC $5 'IU-R', a bare record number
FORM_SUBHEADING = re.compile(r"(?i)^(?:treaties|laws|constitution|statutes|ordinances|proclamations|court rules|charters)\b")
WORK_TITLE = re.compile(r"(?i)\b(?:reports?|series|bulletins?|proceedings|annual|catalogues?|catalogs?|manuscripts?|papers|"
                        r"publications?|circular|memoirs?|transactions|journal|yearbook|directory|minutes|"
                        r"selections|works|poems|letters|[A-Z]\d{3,})\b|\b[A-Z]\d{3,}\b")
_CJK_ROLE = re.compile(r"(?<=[\u3400-\u9fff])(?:共著|共編|共訳|編著|編纂|校訂|校注|校閲|監修|開題|解題|解説|編集|編輯|著|編|訳|譯|撰|輯|纂)$")
ZH_ROLE = re.compile(r",\s*(?:ke|zhu|bian|bian ji|bian zhu|ji|zuan|yi|jiao|shu|hui|chao)\.\s*$")   # case-sensitive
_EPITHET = re.compile(r"(?i)\s+(?:heiliger|heilige|seliger|selige)$")
_NOBLE_OF = re.compile(r"(?i)^(?:(?:duke|duchess|earl|count|countess|marquis|marquess|viscount|baron|baroness|prince|princess|lord|lady)"
                       r"\s+of|(?:duque|duquesa|conde|condesa|marqués|marquesa|vizconde|duc|duchesse|comte|comtesse|marquis|"
                       r"vicomte|duca|conte|marchese|principe)\s+(?:de|d'|di|del|de la))$")
_WIFE = re.compile(r"(?i)^(?:mrs|mme|madame|lady|frau|señora|sra)\.?$")
SEG_SPLIT = re.compile(r"\s*\n\s*|\s{3,}")


def _drop_dup_c1(s):
    out = []
    for i, ch in enumerate(s):
        if "\x80" <= ch <= "\x9f" and i + 1 < len(s):
            try:
                if bytes([ord(ch)]).decode("cp1252") == s[i + 1]:
                    continue
            except UnicodeDecodeError:
                pass
        out.append(ch)
    return "".join(out)


def _fix_controls(s):
    t = repair_mojibake(s)
    t = _drop_dup_c1(t)
    t = _MARC_PAIR.sub(r"\1", t)
    t = _map_c1(t)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"[\t\v\f\x1c-\x1f]+", " ", t)
    t = re.sub(r"[\x00-\x08\x0e-\x1b\x7f-\x9f]", "", t)
    return unicodedata.normalize("NFC", t)


def _prefix_repl(m):
    if not m.group(1):
        return ""
    before = m.string[:m.start()]
    after = m.string[m.end():].split("\x3b")[0]
    if "," not in before.split("\x3b")[-1] and "," not in after:
        return " "
    return m.group(1)


def _drop_identifiers(t):
    t = re.sub(r"(?i)\(\s*(?:https?://)?(?:www\.)?orcid\.org/[^)]*\)", " ", t)
    t = re.sub(r"XX\d\w*?XX\d", " ", t)
    t = ID_ORCID.sub(" ", t)
    lines = []
    for line in t.split("\n"):
        if re.search(r"(?i)\((?:https?://|www\.)[^)]*\)", line):
            lines.append(line)
            continue
        prev = None
        while prev != line:
            prev = line
            line = URL_AT_END.sub("", line)
            line = URL_PREFIX.sub(_prefix_repl, line)
        line = URL_MID.sub(" ", line)  # 'R. , https://orcid.org/... Hertwig' (a mint-time flip of 'Hertwig, R. , URL')
        lines.append(line)
    return "\n".join(lines)


def _strip_punct(x):
    x = x.strip().strip(",\x3b:").strip()
    if x.endswith(".") and not re.search(r"(?:^|[\s.(\-])(?:[^\W\d_]{1,2}|Jr|Sr|Co|Inc|Ltd|Bros|St|Ste|Mme|Mlle|Dr|Mr|Mrs|"
                                          r"Wis|Mass|Ill|Calif|Penn|Va|Md|Mich|Conn|N\.Y|U\.S|D\.C|etc)\.$", x):
        x = x[:-1].rstrip()
    return x


PAREN_ROLE = re.compile(r"(?i)\s*\((?:μτφρ|επιμ|ed|eds|edt|hrsg|trans|tr|transl|comp|ill|illus|coord)\.?\)")


_BNF_ABBREV = frozenset("Ch Th Wm Ph Fr Dr St Ste Mme Mlle Mgr Jos Wilh Chr Aug Alph Alex Ant Edm Ferd Fréd Geo Guill "
                        "Hipp Jac Nic Théod Vict Gust Ern Jn Jh".split())


def _drop_bnf_role(h):
    m = BNF_ROLE.search(h)
    if not m:
        return h
    tok = re.split(r"[\s.,(\-]", h[:m.start()])[-1]
    initial = (len(tok) == 1 and tok.isupper()) or tok in _BNF_ABBREV
    return h[:m.start()] + ("." if initial else "")


def _bare_head(head):
    h = _drop_bnf_role(PAREN_ROLE.sub("", ZH_ROLE.sub("", head)))
    h = INLINE_RELATOR.sub("", h)
    h = PAREN_DATES.sub("", h)          # once the role is gone: 'Campardon, Georges (ca1839-1917). Auteur du texte' (#1944)
    h = INLINE_CENTURY.sub("", h)
    h = INLINE_DATES.sub("", h)
    return INLINE_RELATOR.sub("", h)


def _is_personal(head):
    h = _bare_head(PAREN_DATES.sub("", head)).strip().replace("[", "").replace("]", "").strip()
    h = re.sub(r"\s*\([^)]*\)", "", h)
    parts = [p.strip() for p in h.split(",") if p.strip()]
    if len(parts) < 2:
        return False
    sur, fore = parts[0], parts[1]
    if not _LETTER.search(sur) or not _LETTER.search(fore):
        return False
    if len(parts) >= 3 and sum(1 for x in parts if len(x.split()) >= 2 and x[:1].isupper() and not _RANK.match(x)) >= 3:
        return False       # 'Nivetha Ravishankar*, Dr. Dheepthi Jayamurali, Dr. Pazhanisankar Muthusamy, ...'
    if SUFFIX.match(fore) and len(parts) < 3:
        return False
    if _CORP_WORDS.search(sur) or _CORP_WORDS.search(fore) or len(sur.split()) > 5 or len(fore.split()) > 6:
        return False
    if re.search(r"\d", sur + fore) or "&" in h or re.search(r"(?i)\band\b.*\b(?:company|co)\b", h):
        return False
    return True


def _flip(sur, fore):
    if _CJK.search(sur) and _CJK.search(fore) and not re.search(r"[A-Za-z]", sur + fore):
        return sur + fore
    if re.search(r"(?i)(?:^|\s)[dl]['’]$", fore):     # 'Espinay, Gustave d'' -> 'Gustave d'Espinay' (#1952)
        return f"{fore}{sur}"
    return f"{fore} {sur}"


_DESC_KEEP = re.compile(r"(?i)^(?:al|el|ibn|bin|ben|van|von|de|da|del|della|du|des|di|le|la|dos|das|do|zu|ter|ten|af|av|"
                        r"y|e|und|and|of|the|d['’]|l['’]|al-|el-|abu|bint|binti|bt|bte|degli|dei|vom|zum|zur)\b")
_ADDRESS_WORD = re.compile(r"(?i)^(?:docteur|dr|abbé|abbe|père|frère|soeur|sœur|mère|capitaine|colonel|général|commandant|"
                           r"lieutenant|chanoine|curé|pasteur|rabbin|madame|mme|mademoiselle|mlle|monsieur|m|mgr|"
                           r"monseigneur|professeur|prof|maître|me|veuve|vve)\.?$")


_DESC_NOUN = re.compile(r"(?i)^(?:pape|papa|roi|reine|rex|regina|empereur|impératrice|imperator|évêque|eveque|archevêque|"
                        r"episcopus|archiepiscopus|cardinal|abbesse|prieur|prieure|grand|chanoine|curé|vicaire|moine|"
                        r"religieux|religieuse|dominicain|franciscain|jésuite|bénédictin|carme|capucin|chartreux|poète|"
                        r"poeta|médecin|chirurgien|avocat|notaire|peintre|sculpteur|graveur|imprimeur|libraire|éditeur|editeur|photographe|"
                        r"professeur|ambassadeur|consul|ministre|gouverneur|intendant|maréchal|amiral|officier|"
                        r"lieutenant|ingénieur|architecte|historien|philosophe|théologien|fils|fille|veuve|"
                        r"provost|bishop|archbishop|king|queen|emperor|abbot|abbess|prior|monk|saint|santo|santa|"
                        r"infant|infante|infanta|sultan|calife|khan|émir|patriarche|patriarch)$")
_NOBLE_WORD = re.compile(r"(?i)^(?:comte|comtesse|baron|baronne|marquis|marquise|vicomte|vicomtesse|duc|duchesse|prince|"
                         r"princesse|chevalier|seigneur|sieur|abbé|abbe|mère|mere|père|pere|fray|frère|sœur|soeur|dom|"
                         r"don|doña|sister|brother|father|mother)$")


def _is_descriptor(fore):
    w = fore.split()
    if not w or not w[0][:1].islower() or _NOBLE_WORD.match(w[0]) or (len(w) == 1 and _ADDRESS_WORD.match(w[0])):
        return False
    if _DESC_KEEP.match(w[0]):
        return False
    return bool(_DESC_NOUN.match(w[0]) or (len(w) >= 3 and all(x[:1].islower() for x in w[:2])
                                           and re.search(r"(?i)\s(?:de|d['’]|du|des|of|von|di|del|da)\s*[A-ZÀ-ÖØ-Þ]", fore)))


_PRO_TITLE = re.compile(r"^(?:Docteur|Dr\.?|Monseigneur|Mgr\.?|Professeur|Prof\.?|Abbé|Père|Frère|Chanoine|"
                        r"Capitaine|Colonel|Général|Commandant|Lieutenant)\s+(?=[A-ZÀ-ÖØ-Þ][^\s.]*[a-zà-ÿ])")


def _personal(segs):
    head = segs[0]
    suffix = address = noble = ""
    for s in segs[1:3]:
        s2 = s.strip().strip(",").strip()
        first = s2.split(",")[0].strip()
        if SUFFIX.match(first):
            suffix = first
            continue
        if _WIFE.match(first):
            address = first
            continue
        if _NOBLE_OF.match(first):
            noble = first
            continue
        if not (DATE_SEG.match(s2) or RELATOR_SEG.match(s2) or s2.startswith("(")):
            break
    firm = FIRM_PAREN.search(head) or next((m for m in map(FIRM_PAREN.search, segs[1:3]) if m), None)
    h = _bare_head(PAREN_DATES.sub("", head))
    paren_fore = re.match(r"^([^,()]+?)\s*\(([^()]+)\)\s*,", h)
    paren_fore = paren_fore if paren_fore and _forename_shaped(paren_fore.group(1), paren_fore.group(2)) else None
    h = re.sub(r"\s*\([^)]*\)", "", h)   # fuller form / religious order / maiden name in parentheses
    h = h.replace("[", "").replace("]", "")
    parts = [p.strip() for p in h.split(",")]
    parts = [p for p in parts if p]
    if len(parts) < 2:
        return _strip_punct(parts[0]) if parts else None
    sur, fore, rest = parts[0], parts[1], parts[2:]
    if SUFFIX.match(fore) and rest:      # 'Lucas, Jr., Fielding'
        suffix, fore, rest = fore, rest[0], rest[1:]
    for r in rest:  # 'Towle, Charles C., Jr.'  'Bismarck, Johanna, Fürstin von' (a title: dropped)
        if SUFFIX.match(r):
            suffix = suffix or r
        elif _WIFE.match(r):
            address = address or r
        elif _NOBLE_OF.match(r):
            noble = noble or r
    sur, fore = _strip_punct(sur), _strip_punct(_EPITHET.sub("", fore))
    sur, fore = _CJK_ROLE.sub("", sur), _CJK_ROLE.sub("", fore)
    if not sur or not fore:
        return None
    if _is_descriptor(fore):   # 'Billot, médecin de l'Empereur'  'Florainville (Christine de), abbesse de ...' (#1944)
        pf = paren_fore.group(2).strip() if paren_fore else ""
        if pf[:1].isupper() and not re.search(r"\d", pf) and not re.fullmatch(r"(?i)s\.?|saint|sainte|ste?\.?", pf):
            return _flip(sur, _strip_punct(pf))
        return sur
    fore = _PRO_TITLE.sub("", fore)   # 'Flórez, Docteur Ricardo L.' -> 'Ricardo L. Flórez', 'Abbé de' and 'Père' stay
    if re.match(r"(?:of|da|de|di|du|del|della|von|van|zu)\s", fore):   # 'Mary, of Modena, Queen' 'Cino, da Pistoia'
        name = f"{sur} {fore}"
    elif _RANK.match(fore) or re.search(r"\s[IVX]+$", sur):   # 'Aaron ben Jacob, The great'  'Manouēl II, Palaiologos'
        name = sur
    elif noble:                                        # 'Newcastle, Thomas Pelham-Holles, Duke of'
        name = f"{fore}, {noble} {sur}"
    else:                                              # forename, or a term of address: 'Mrs. Oliphant', 'Sister Mary Xavier'
        name = _flip(sur, fore)
    if firm:                             # 'Barco López, Plácido (Viuda de)': the printer's widow, a firm
        name = f"{name} {firm.group(0)}"
    if address and not name.lower().startswith(address.lower().rstrip(".")):   # 'West, Max, Mrs.' -> 'Mrs. Max West'
        address = address if address.endswith(".") or len(address) > 4 else address + "."
        name = f"{address} {name}"
    if suffix:
        suffix = suffix.rstrip(",")
        suffix = suffix if suffix.endswith(".") or re.fullmatch(r"(?i)[ivx]+|\d\w\w", suffix) else suffix + "."
        name = f"{name} {suffix}"
    return name


def _corporate(segs):
    keep = []
    for i, s in enumerate(segs):
        s2 = s.strip()
        if not s2:
            continue
        if i > 0 and (RELATOR_SEG.match(s2) or FORM_SUBHEADING.match(s2)
                      or (WORK_TITLE.search(s2) and not _CORP_WORDS.search(s2))):
            nxt = segs[i + 1].strip() if i + 1 < len(segs) else ""
            if keep and re.fullmatch(r"\([^()\d]+\)\.?", nxt) and RELATOR_SEG.match(s2):   # a place after a role
                keep[-1] = keep[-1].rstrip(", ")
                keep.append(nxt.rstrip("."))
            break
        if i > 0 and re.fullmatch(r"[A-Z]{2,5}", s2) and re.search(r"(?i)collection|library|\)$", keep[-1] if keep else ""):
            continue   # MARC $5 holding code after a collection ('... (Library of Congress) DLC')
        if i > 0 and (HOLDING_CODE.fullmatch(s2.rstrip(".")) or SHELFMARK.fullmatch(s2)
                      or (re.fullmatch(r"\d{4}", s2) and not 1000 <= int(s2) <= 2030)):
            continue
        r = INLINE_RELATOR.sub("", s2)
        keep.append(r.strip())
        if r != s2:
            break
    if not keep:
        return None
    out = _drop_bnf_role(ZH_ROLE.sub("", " ".join(keep))).strip()
    return out.rstrip(",\x3b: ").strip()


_PAREN_TYPE = re.compile(r"(?i)\b(?:périodique|revue|journal|duché|royaume|comté|principauté|province|département|"
                         r"région|ville|firme|société|collectif|groupe|association|ordre|congrégation|famille|maison|"
                         r"dynastie|seigneurie|abbaye|diocèse|paroisse|commune|canton|pays|état|empire|république)\b")


def _forename_shaped(sur, fore):
    sw = [x for x in sur.split() if not _DESC_KEEP.match(x) and x.lower() not in ("la", "le", "les", "l'")]
    fw = [x for x in re.split(r"[\s,]+", fore.strip()) if x and not _DESC_KEEP.match(x)]
    if not sw or not fw or len(sw) > 2 or len(fw) > 5 or _PAREN_TYPE.search(fore) or _CORP_WORDS.search(sur):
        return False
    if any(x[:1].islower() for x in sw):
        return False
    return all(x[:1].isupper() for x in fw if not _NOBLE_WORD.match(x))


_BNF_OLD = re.compile(r"^(?P<sur>[A-ZÀ-ÖØ-Þ][^,()\d]{0,40}?)\s*\((?P<fore>[A-ZÀ-ÖØ-Þ][^(),\d]{0,60}?)"
                      r"(?:,\s*(?P<noble>(?:marquis|marquise|duc|duchesse|comte|comtesse|vicomte|vicomtesse|baron|baronne|"
                      r"prince|princesse|seigneur|sieur|chevalier)\s+(?:de|d['’]|du|des)))?\)$")


_ADDR_IN_DATES = re.compile(r"(?i)^(?P<sur>[^\W\d_][^\s(),.]*)\s*\([^()]*\x3b\s*(?P<addr>abbé|abbe|père|frère|soeur|sœur|mère|"
                            r"docteur|curé|chanoine|pasteur|capitaine|colonel|général|commandant|lieutenant|madame|"
                            r"mademoiselle|veuve|dame)\)$")


def _plain(line):
    h = _drop_bnf_role(line)
    if h != line:   # old BnF form 'Sourdis (Charles d'Escoubleau, marquis de). Auteur du texte' (#1944)
        m = _BNF_OLD.match(h.strip())
        inner = m.group("fore").strip() if m else ""
        sure = m and (m.group("noble") or re.search(r"(?i)\s(?:de|d['’]|du|des)$", inner)
                      or re.fullmatch(r"(?:[A-ZÀ-ÖØ-Þ]\.[\s-]?)+", inner) or _ADDRESS_WORD.match(inner))
        if sure and _forename_shaped(m.group("sur"), inner) \
                and not re.match(r"(?i)mus[ée]e|archives|biblioth|[ée]cole|lyc[ée]e|coll[èe]ge|h[ôo]pital|mairie|ville|"
                                 r"commune|paroisse|abbaye|[ée]glise|chambre|syndicat|parti|ordre|loge|soci", m.group("sur")):
            sur = m.group("sur").strip()
            return f"{inner}, {m.group('noble')} {sur}" if m.group("noble") else f"{inner} {sur}"
        m = _ADDR_IN_DATES.match(h.strip())
        if m:   # 'Fangouse (17..-17.. , abbé)': no forename, so the term of address stays: 'abbé Fangouse' (#1952)
            return f"{m.group('addr')} {m.group('sur')}"
    h = INLINE_RELATOR.sub("", h)
    if not _CORP_WORDS.search(h):
        h = PAREN_DATES.sub("", h)
        h = INLINE_CENTURY.sub("", h)
        h = INLINE_DATES.sub("", h)
    h = re.sub(r"\s*\((?:Prof|Dr|PhD|Ph\.D|MD|Dipl)\.?[^)]*\)|\s*\(\?\)", "", h)   # '(?)': BnF's doubt mark (#1952)
    h = _CJK_ROLE.sub("", h.strip().rstrip(".,")) if _CJK_ROLE.search(h.strip().rstrip(".,")) else h
    if not h.strip():
        return None
    return _strip_punct(h) if h != line else line.strip()


def _join_parens(t):
    out, depth = [], 0
    for ch in t:
        depth += (ch == "(") - (ch == ")" and depth > 0)
        out.append(" " if depth and ch in "\n\t " else ch)
    return re.sub(r"\(\s+", "(", re.sub(r" {2,}", " ", "".join(out))) if "(" in t else t


def _unflip(t):
    m = re.search(r"(?:^|\s)880-\d{2}\.?(?=\s|$)", t)
    if m and m.start() > 0 and t[:m.start()].strip() and t[m.end():].strip():
        y = t[:m.start()].strip()
        return t[m.start():].strip() + (",\n" if "\n" in y else ", ") + y
    m = _flipped_bnf(t)
    if m:                                    # 'Georges. Auteur du texte Mazet' was 'Mazet, Georges. Auteur du texte' (#1952)
        fore = m.group("fore").strip()
        if fore in _DEPARTEMENTS:            # a commune: 'Mespaul (Finistère)'
            return f"{m.group('sur').strip()} ({fore}). {m.group('role')}"
        return f"{m.group('sur').strip()}, {fore}. {m.group('role')}"
    m = re.match(r"^(\S+\)\.?)\s+(.*\([^()]*)$", t, re.S)
    if m and "(" not in m.group(1):
        return m.group(2).rstrip() + ", " + m.group(1)
    m = re.match(rf"^([^\d\n,]+?)\s+({_YEAR}\s*-\s*(?:{_YEAR})?)\s+([^\W\d][^\s\d,]*)$", t)
    if m:                                    # 'Johann Christoph von 1747-1827 Zabuesnig' was 'Zabuesnig, Johann ... 1747-1827'
        return f"{m.group(3)}, {m.group(1)} {m.group(2)}"
    m = re.match(rf"^({_YEAR}\s*-\s*(?:{_YEAR})?\.?)\s+([^\W\d][^\d,]*)$", t)
    if m:                                    # '1870-1916. Saki' was 'Saki, 1870-1916.'
        return f"{m.group(2)}, {m.group(1)}"
    m = re.match(rf"(?i)^((?:{_RELATORS})\.?)\s+([^\W\d][^\s\d,]*)$", t)
    if m:
        return f"{m.group(2)}, {m.group(1)}"
    m = re.match(r"^etc\.\s+(.+)$", t, re.S)
    if m:                                    # 'etc. Switzerland.\n Treaties' was 'Switzerland. Treaties, etc.'
        return m.group(1) + ", etc."
    return t


def _split_people(t):
    if "(" not in t:
        return t.split("\x3b")
    out, cur, depth = [], [], 0
    for ch in t:
        depth += (ch == "(") - (ch == ")" and depth > 0)
        if ch == "\x3b" and depth == 0:
            out.append("".join(cur))
            cur = []
        else:
            cur.append(ch)
    out.append("".join(cur))
    return out


_WIDE_RUN = re.compile(r"(\n| {3,})")
_FR_ROLE_WORD = re.compile(rf"(?i)^(?:{'|'.join(_BNF_ROLE_WORDS)})\b")


def _join_name_runs(t):
    parts = _WIDE_RUN.split(t) if "   " in t else [t]
    if len(parts) < 3:
        return t
    prev, sep, s2 = parts[-3], parts[-2], parts[-1].strip()
    if (sep != "\n" and (re.search(r"[^\W\d_]$", prev)    # or right after 'Surname,': 'YU,   CHIH-AI'
                          or len(parts) == 3 and re.fullmatch(r"(?=[^,\n]*[^\W\d_])[^,\n]*,", prev.strip()))
            and re.match(r"[^\W\d_]", s2) and len(s2.split()) <= 3
            and "," not in s2 and "." not in s2 and not (DATE_SEG.match(s2) or RELATOR_SEG.match(s2)
                                                        or FORM_SUBHEADING.match(s2) or WORK_TITLE.search(s2)
                                                        or _CORP_WORDS.search(s2) or _FR_ROLE_WORD.match(s2)
                                                        or URI.search(s2) or SUFFIX.match(s2) or _ADDRESS.match(s2))):
        return "".join(parts[:-2]) + " " + parts[-1]
    return t


def _clean_one(t):
    t = _join_name_runs(t)
    t = "\n".join(_join_parens(x) if "(" in x else x for x in SEG_SPLIT.split(t.strip()))
    segs = [s for s in t.split("\n") if s.strip()]
    if not segs:
        return None
    segs = [MARC880.sub(" ", s).strip() for s in segs]
    segs = [s for i, s in enumerate(segs) if s and not (i == 0 and re.fullmatch(r"\d{3,}\.?", s))]
    if not segs:
        return None
    segs[0] = re.sub(r"(?i)^(?:[\w-]+\s+){0,3}(?:of|for|to|on)\s*\((?:work|expression|manifestation)\)\s*:\s*", "",
                     segs[0])
    segs[0] = re.sub(r"(?i)^(?:created|made|designed|drawn|painted|photographed|engraved|printed|published)\s+by\s+", "",
                     segs[0])
    segs[0] = re.sub(r"(?i)^(?:adaptation of|based on|continuation of|translation of|parody of|"
                     r"imitation of|sequel to|supplement to)\s*:\s*", "", segs[0])
    if not segs[0]:
        segs = segs[1:]
        if not segs:
            return None
    stop = next((i for i, x in enumerate(segs[1:], 1) if DATE_SEG.match(x.strip()) or RELATOR_SEG.match(x.strip())),
                len(segs))
    unit_follows = any(_CORP_WORDS.search(x) and len(x.split()) <= 5 for x in segs[1:stop])
    if len(segs) > 1 and "," in segs[0] and re.fullmatch(r"(?:[A-Z]\.\s?)+", segs[1].strip()):
        segs = [f"{segs[0].rstrip()} {segs[1].strip()}"] + segs[2:]          # 'Saccardo, P        A.'
    if (_is_personal(segs[0]) and len(segs) > 2 and not any(DATE_SEG.match(x.strip()) or RELATOR_SEG.match(x.strip())
                                                            for x in segs)
            and sum(1 for x in segs[1:] if _is_personal(x) and len(x.split()) <= 5) >= 2):
        return "\x3b ".join(_personal([x]) or x.strip() for x in segs)
    if _is_personal(segs[0]) and not unit_follows:
        return _personal(segs)
    if len(segs) > 1 and not _CORP_WORDS.search(segs[0]) and not unit_follows:
        if re.fullmatch(r"[IVX]+,?", segs[1].strip()):   # numeration ($b): 'Manouēl    II,'
            segs = [f"{segs[0].strip()} {segs[1].strip()}"] + segs[2:]
        cut = next((i for i, x in enumerate(segs[1:], 1) if DATE_SEG.match(x.strip())), None)
        if cut is not None and segs[0].rstrip().endswith(","):   # a personal $a ends with a comma, a corporate one with '.'
            line = ", ".join(x.strip().rstrip(",") for x in segs[:cut + 1])
            if _is_personal(line):
                return _personal([line])
            return _plain(line)
    if len(segs) == 1:
        return _plain(segs[0])
    return _corporate(segs)


def _squeeze(t):
    t = t.replace("\t", " ").replace("\\t", " ").replace("\\n", "\n")
    t = re.sub(r"[ \u00a0]*\n(?:[ \u00a0]*\n)*[ \u00a0]{2,}", "\ue000", t)   # subfield break (placeholder)
    lines = [x.strip() for x in re.split(r"\n+", t) if x.strip()]
    t = lines[0] if lines else ""
    for x in lines[1:]:   # a bare line break: two names, or a break inside one ('Nguyễn\n Văn Luyện')
        prev = t.split("\ue000")[-1].split("\x3b")[-1]
        both_names = all(len(y.split()) >= 2 and len(y) >= 8 for y in (prev, x.split("\ue000")[0]))
        t += ("\x3b " if both_names else " ") + x
    t = t.replace("\ue000", "\n")
    t = re.sub(r"(?:\x3b\s*){2,}", "\x3b ", t)
    return re.sub(r"[ \u00a0]{3,}", "   ", t).strip().strip("\x3b").strip()


def clean_byline_name(s):
    return _clean_pass(s)


def _clean_pass(s):
    if not s or not isinstance(s, str) or not has_artefact(s[:2000]):
        return s
    t = _fix_controls(s) if CTRL.search(s) else s
    t = unicodedata.normalize("NFC", t)
    structural = bool(URI.search(t) or MARC880.search(t) or DATES_END.search(t) or WIDE.search(t) or "\n" in t.rstrip()
                      or IDORCID.search(t) or BNF_ROLE.search(t) or FLIPPED_DATES.search(" ".join(t.split()))
                      or _flipped_bnf(t)
                      or re.match(r"\s*\d{3,4}\??\s*-", t))
    t = _squeeze(t)
    if not structural:  # rule 6: only control characters were wrong (a trailing role word goes too)
        parts = [" ".join(p.split()) for p in t.split("\x3b")]
        parts = [_strip_punct(INLINE_RELATOR.sub("", PAREN_ROLE.sub("", p))) if INLINE_RELATOR.search(p) or
                 PAREN_ROLE.search(p) else p for p in parts]
        out = "\x3b ".join(p for p in parts if p)
        return out if out and _LETTER.search(out) else s
    if "&" in t:
        t = html.unescape(t)
    t = REPEC.sub(" ", EMAIL.sub(" ", t))
    t = _drop_identifiers(t)
    t = _unflip(t)
    parts = [p.strip() for p in _split_people(t)]
    parts = [p for p in parts if p and not SHELFMARK.fullmatch(" ".join(p.split()))]
    if len(parts) > 1 and all(_LETTER.search(p) for p in parts):
        cleaned = [_clean_one(p) for p in parts]
        out = "\x3b ".join(c for c in cleaned if c)
    else:
        out = _clean_one(parts[0]) if parts else None
    if not out or LABEL_ONLY.match(out):
        return s
    if " ".join(out.split()) == " ".join(s.split()) and not CTRL.search(s) and not re.search(r"[.,):]\s{3,}|\s{3,}\(", s):
        return s
    out = " ".join(out.split())
    if not _LETTER.search(out) or has_artefact(out) and (CTRL.search(out) or URI.search(out) or MARC880.search(out)):
        return s
    return out


return clean_byline_name(s)
$$

In [ ]:
-- oxjob #1912: the byline names to serve in place of library-catalogue strings. Built every run from the current
-- profiles and works base, so new catalogue records are cleaned the night they arrive.
-- 1. work_byline_name_fixes: every name a served byline can show (a profile's display name, or the source record's
--    name on a seat with no profile) that holds a catalogue artefact, with the name it holds. Only changed names.
--    The RLIKE gate is a superset of the function's own test (control characters, URLs, 880 tags, life dates,
--    3+ spaces, ORCID ids, BnF roles, the mint-time flip 'Dan 1964- Porat'). The function returns its input otherwise.
-- 2. work_byline_author_name_fixes: the same, per profile (COALESCE(openalex_authors, authors) display name, exactly
--    as author_data_enriched below reads it), so the main statement joins on author_id like its other joins.
-- 3. work_byline_fallback_name_fixes: per work, a map from a seat's source name to its clean name, for seats with no
--    profile (the display_name fallback).
-- Revert: remove this cell, the function cell above it, and the three oxjob #1912 edits in the main statement.
CREATE OR REPLACE TABLE openalex.works.work_byline_name_fixes AS
WITH names AS (
  SELECT DISTINCT n FROM (
    SELECT display_name AS n FROM openalex.authors.openalex_authors
    UNION ALL
    SELECT display_name AS n FROM openalex.authors.authors
    UNION ALL
    SELECT a.author.display_name AS n FROM openalex.works.openalex_works_base LATERAL VIEW EXPLODE(authorships) t AS a
  )
  WHERE n RLIKE '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]|(?i)(https?://|www\\.|id\\.loc\\.gov|viaf\\.org/|orcid\\.org/|lattes\\.cnpq\\.br/|id_orcid)|(^|\\s)880-[0-9]{2}|[0-9]{3,4}\\??[\\s\\x{00A0}]*-[\\s\\x{00A0}]*([0-9]{3,4}\\??)?[\\s\\x{00A0}]*[.,)]?[\\s\\x{00A0}]*$|[0-9]{3,4}\\??\\s*-\\s*([0-9]{3,4}\\??)?\\s+\\p{L}|\\S[\\s\\x{00A0}]{3,}\\S|[0-9]{4}- ?[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]|\\.\\s*(auteur|éditeur|traducteur|illustrateur|préfacier|compositeur|librettiste|graveur|photographe|cartographe|imprimeur|libraire|dessinateur|annotateur|commentateur|rédacteur)'
)
SELECT raw_name, clean_name FROM (
  SELECT n AS raw_name, openalex.authors.clean_byline_name(n) AS clean_name FROM names
) WHERE clean_name IS DISTINCT FROM raw_name;

CREATE OR REPLACE TABLE openalex.works.work_byline_author_name_fixes AS
SELECT p.id AS author_id, f.clean_name
FROM (
  SELECT COALESCE(oa.id, ar.id) AS id, COALESCE(oa.display_name, ar.display_name) AS display_name
  FROM openalex.authors.openalex_authors oa
  FULL OUTER JOIN openalex.authors.authors ar ON oa.id = ar.id
) p
JOIN openalex.works.work_byline_name_fixes f ON f.raw_name = p.display_name;

CREATE OR REPLACE TABLE openalex.works.work_byline_fallback_name_fixes AS
SELECT b.work_id, MAP_FROM_ENTRIES(COLLECT_SET(STRUCT(f.raw_name, f.clean_name))) AS name_fix
FROM (
  SELECT id AS work_id, a.author.display_name AS n
  FROM openalex.works.openalex_works_base LATERAL VIEW EXPLODE(authorships) t AS a
  WHERE a.author.display_name RLIKE '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]|(?i)(https?://|www\\.|id\\.loc\\.gov|viaf\\.org/|orcid\\.org/|lattes\\.cnpq\\.br/|id_orcid)|(^|\\s)880-[0-9]{2}|[0-9]{3,4}\\??[\\s\\x{00A0}]*-[\\s\\x{00A0}]*([0-9]{3,4}\\??)?[\\s\\x{00A0}]*[.,)]?[\\s\\x{00A0}]*$|[0-9]{3,4}\\??\\s*-\\s*([0-9]{3,4}\\??)?\\s+\\p{L}|\\S[\\s\\x{00A0}]{3,}\\S|[0-9]{4}- ?[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]|\\.\\s*(auteur|éditeur|traducteur|illustrateur|préfacier|compositeur|librettiste|graveur|photographe|cartographe|imprimeur|libraire|dessinateur|annotateur|commentateur|rédacteur)'
) b
JOIN openalex.works.work_byline_name_fixes f ON f.raw_name = b.n
GROUP BY b.work_id

In [ ]:
-- oxjob #1642: cap the main statement below at 12 min. On 8, 9 and 10 Nov it froze inside the nightly: one task of its
-- last stage (the authorships arrays, ~3,080 tasks) failed and never re-ran, so the statement sat until the task timeout
-- (25 min, then the retry on the same warehouse cluster froze too). Healthy runs take 3.7-4.4 min. A timed-out statement
-- fails the task, and the task's retry (min_retry_interval_millis 150000 in jobs/walden_end2end.yaml) waits until
-- PROD-WH (auto-stop 1 min) has stopped, so it gets a fresh cluster. Revert: delete this cell.
SET STATEMENT_TIMEOUT = 720

In [ ]:
-- oxjob #1642 shepherd, night 11-16, TEMPORARY (revert right after run 263504769050302): the main statement froze 3 times
-- on jobs-4xl-authorships (04:21, 04:46, 05:08 CT; ~21 task-h each, STATEMENT_TIMEOUT 720). Its exact output (statement
-- sha 7e9ccf861e12) was built on desk-4xl in 5.6 min and loaded into openalex.works.work_authorships (version 2570,
-- 410,459,886 rows). This cell only checks it. Revert: git revert <this commit>.
SELECT CASE WHEN (SELECT count(*) FROM openalex.works.work_authorships) < 400000000
            THEN raise_error('work_authorships has under 400M rows: the shepherd load is missing')
            ELSE 'ok' END AS shepherd_check